In [10]:

import os
import random
import copy
import warnings
from pathlib import Path


import numpy as np
import pandas as pd


from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader
)

# ----------------------------
# Torchvision
# ----------------------------
from torchvision import (
    transforms,
    models
)

from sklearn.model_selection import (
    train_test_split
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    hamming_loss,
    multilabel_confusion_matrix
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("All standard libraries imported successfully.")


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

ModuleNotFoundError: No module named 'tqdm'

In [1]:
from ultralytics import YOLO

print("Ultralytics imported successfully")

Ultralytics imported successfully


In [2]:
from ultralytics import YOLO

model = YOLO("yolo11n-cls.pt")

print("YOLO classification model loaded successfully.")

YOLO classification model loaded successfully.


In [3]:
print(model.model)

ClassificationModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)


In [5]:
import os

In [6]:
EXCEL_PATH = r"C:\Users\minsa_\Downloads\Disease_Labels_Final_filled.xlsx"

IMAGE_DIR = r"C:\Users\minsa_\OneDrive\Desktop\Disease_Cell_Images"

print("Excel exists:", os.path.exists(EXCEL_PATH))
print("Image folder exists:", os.path.exists(IMAGE_DIR))

Excel exists: True
Image folder exists: True


In [11]:
df = pd.read_excel(EXCEL_PATH)

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (1054, 9)

Columns:
['Tissue_ID', 'Eye', 'Image_ID', 'Disease_Label', 'Disease_Severity', 'ECD_CD_cells_per_mm2', 'CV', 'HEX_percent', 'SD']


In [12]:
df.head()

,Tissue_ID,Eye,Image_ID,Disease_Label,Disease_Severity,ECD_CD_cells_per_mm2,CV,HEX_percent,SD
0,NEB-2020-09-12654 R,Right,NEB-2020-09-12654 R_SP,Normal,NaN,2132.0,33.0,58.0,153.0
1,NEB-2020-09-12655 L,Left,NEB-2020-09-12655 L_SP,Folds,Moderate,2212.0,33.0,47.0,147.0
2,NEB-2020-09-12656 R,Right,NEB-2020-09-12656 R_SP,Normal,NaN,2538.0,35.0,53.0,137.0
3,NEB-2020-09-12657 L,Left,NEB-2020-09-12657 L_SP,Folds,Mild,2584.0,29.0,69.0,112.0
4,NEB-2020-09-12658 R,Right,NEB-2020-09-12658 R_SP,Normal,NaN,3115.0,38.0,49.0,122.0


In [13]:
image_files = [
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("Images in folder:", len(image_files))

print("\nFirst 10 images:")
print(image_files[:10])

Images in folder: 1042

First 10 images:
['NEB-08-16298 L_SP.jpg', 'NEB-2011-09-14230 R_SP.jpg', 'NEB-2020-09-12654 R_SP.jpg', 'NEB-2020-09-12655 L_SP.jpg', 'NEB-2020-09-12656 R_SP.jpg', 'NEB-2020-09-12657 L_SP.jpg', 'NEB-2020-09-12658 R_SP.jpg', 'NEB-2020-09-12659 L_SP.jpg', 'NEB-2020-09-12660 L_SP.jpg', 'NEB-2020-09-12661 R_SP.jpg']


In [14]:
image_set = set(image_files)

df["Image_File"] = df["Image_ID"].astype(str) + ".jpg"

df["Image_Exists"] = df["Image_File"].isin(image_set)

print("Total Excel records:", len(df))
print("Matching images:", df["Image_Exists"].sum())
print("Missing images:", (~df["Image_Exists"]).sum())

print("\nMissing image examples:")
print(
    df.loc[~df["Image_Exists"], ["Image_ID", "Disease_Label"]]
    .head(20)
)

Total Excel records: 1054
Matching images: 1040
Missing images: 14

Missing image examples:
                   Image_ID                                      Disease_Label
77                      NaN                                             Normal
173                     NaN                                             Normal
187                     NaN                                             Normal
284                     NaN                                             Normal
285                     NaN                                             Normal
298                     NaN                                             Normal
322                     NaN                                             Normal
362                     NaN                                             Normal
453                     NaN                                             Normal
616  NEB-2023-02-14710 R_SP  FECD; Polymorphism and pleomorphism; Low cell ...
729  NEB-2023-04-14937 R_SP            

In [15]:
print(df["Disease_Label"].value_counts(dropna=False))

Disease_Label
Normal                                                                   504
Folds                                                                    324
FECD; Polymorphism and pleomorphism; Low cell density                     88
Guttata                                                                   53
Guttata; Folds                                                            28
FECD; Polymorphism and pleomorphism; Low cell density; Folds              20
FECD; Polymorphism and pleomorphism; Low cell density; Guttata             8
Polymorphism and pleomorphism                                              5
Poor cornea                                                                5
Folds; Guttata                                                             3
FECD; Polymorphism and pleomorphism; Low cell density; Guttata; Folds      3
Guttata; FECD; Polymorphism and pleomorphism; Low cell density             2
FECD; Polymorphism and pleomorphism                           